# 05 — Extracción de características (v2)

## Objetivo

Extraer características separando **tejido pulmonar**, **forma de la máscara**
y **contexto/fuente**, para poder demostrar qué parte del rendimiento se debe
a cada una. La matriz se generó con `scripts/rebuild_features.py` sobre las
8.256 imágenes únicas.

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1. Matriz de características

In [ ]:
feat = pd.read_csv(DATA / "features_tbx11k_v2.csv")
print("Shape:", feat.shape)
print(feat["class_name"].value_counts().to_string())
print("\nNulos:", int(feat.drop(columns=["image_path", "class_name", "label"]).isnull().sum().sum()))
print("Infinitos:", int(np.isinf(feat.drop(columns=["image_path", "class_name", "label"]).values).sum()))
manifest = json.loads((DATA / "features_v2_manifest.json").read_text(encoding="utf-8"))
manifest

## 1b. Cadena de archivos

`df_base.csv` (01) → `df_model.csv` (deduplicación exacta, generada por `scripts/build_model_meta.py` o por esta etapa) → `features_tbx11k_v2.csv` (features de las 8.256 imágenes).


In [ ]:
model_meta = pd.read_csv(DATA / "df_model.csv")
print("df_model:", len(model_meta))
print(model_meta["class_name"].value_counts().to_string())
print("Coincide con la matriz de features:",
      set(model_meta["image_path"]) == set(feat["image_path"]))


## 2. Familias de características

| Familia | Qué mide | Uso |
|---|---|---|
| `texture` (96) | intensidad, GLCM, LBP y Gabor **dentro del campo pulmonar** | modelo principal |
| `hog` (1.764) | gradientes orientados sobre el recorte torácico | comparación e interpretabilidad espacial |
| `shape` (64) | geometría de la máscara + miniatura 8×8 | **solo control de sesgo** |
| `context` (352) | textura del fondo + miniatura 16×16 | **solo control de fuente** |
| `quality` (4) | área, balance, contraste y fallback | diagnóstico de segmentación |

La textura usa la máscara como soporte: el exterior se rellena con la mediana
pulmonar (no con 0) para no crear bordes artificiales. HOG se calcula sobre el
recorte completo, sin máscara, para que no codifique el contorno.

In [ ]:
from src.features import get_feature_groups
groups = get_feature_groups(feat)
pd.DataFrame({"familia": list(groups), "n_features": [len(v) for v in groups.values()]})

## 3. Contraste robusto: se elimina el artefacto `ID_contrast`

In [ ]:
feat[["mean", "std", "contrast", "ID_contrast", "SI_contrast"]].describe().round(2)

El contraste ahora es `p95 − p5` en lugar de `max − min`. El antiguo
`ID_contrast ≈ 245` era un artefacto de CLAHE saturado; ahora los valores son
interpretables y no dependen de píxeles extremos.

## 4. Distribuciones GLCM por clase

In [ ]:
from src.visualization import plot_glcm_distributions, plot_lbp_histograms
plot_glcm_distributions(feat)

## 5. Histogramas LBP promedio

In [ ]:
plot_lbp_histograms(feat)

## 6. Calidad de la máscara por clase

In [ ]:
feat.groupby("class_name")[["mask_area_fraction", "mask_lr_balance",
                            "mask_contrast", "mask_fallback"]].mean().round(3)

## Conclusiones

- La matriz tiene 2.290 características, sin nulos ni infinitos.
- El modelo principal usará solo `texture` (96), una decisión deliberada:
  HOG/PHOG ya no dominan la selección y la interpretabilidad es directa.
- `shape` y `context` existen para cuantificar el atajo, no para mejorar el
  modelo.
- La extracción completa tardó ~6.7 min con 8 procesos sobre 8.256 imágenes.

**Siguiente etapa:** `06_Seleccion_Caracteristicas.ipynb`.